# Homework 3 — Module 3 Modeling (2026 cohort)
Pipeline reproducible sobre `data.parquet` (33 tickers US/EU/India), `Date >= 2000-01-01`, target `is_positive_growth_30d_future`, split temporal 70/15/15, árboles con `random_state=42`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier, export_text

DATA = Path("data.parquet")
TARGET = "is_positive_growth_30d_future"
pd.options.mode.chained_assignment = None

In [2]:
df_full = pd.read_parquet(DATA)
df = df_full[df_full["Date"] >= "2000-01-01"].copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)
df["ln_volume"] = np.where(df["Volume"] > 0, np.log(df["Volume"]), np.nan)
print("rows:", len(df), "| dates:", df["Date"].min(), "->", df["Date"].max(), "| tickers:", df["Ticker"].nunique())

GROWTH = [g for g in df.keys() if g.startswith("growth_") and ("future" not in g)]
TECH_WANT = ["adx","adxr","apo","aroon_1","aroon_2","aroonosc","bop","cci","cmo","dx","macd","macdsignal","macdhist","macd_ext","macdsignal_ext","macdhist_ext","macd_fix","macdsignal_fix","macdhist_fix","mfi","minus_di","mom","plus_di","dm","ppo","roc","rocp","rocr","rocr100","rsi","slowk","slowd","fastk","fastd","fastk_rsi","fastd_rsi","trix","ultosc","willr","ad","adosc","obv","atr","natr","ht_dcperiod","ht_dcphase","ht_phasor_inphase","ht_phasor_quadrature","ht_sine_sine","ht_sine_leadsine","ht_trendmod","avgprice","medprice","typprice","wclprice"]
TECH = [c for c in TECH_WANT if c in df.columns]
PAT = [g for g in df.keys() if g.startswith("cdl")]
CUST = [c for c in ["SMA10","SMA20","growing_moving_average","high_minus_low_relative","volatility","ln_volume"] if c in df.columns]
MACRO = [c for c in ["gdppot_us_yoy","gdppot_us_qoq","cpi_core_yoy","cpi_core_mom","FEDFUNDS","DGS1","DGS5","DGS10"] if c in df.columns]
NUMERICAL = GROWTH + TECH + PAT + CUST + MACRO
print("NUMERICAL:", len(NUMERICAL), "| GROWTH:", len(GROWTH), "| TECH:", len(TECH), "| PAT:", len(PAT), "| CUST:", CUST, "| MACRO:", MACRO)

rows: 191465 | dates: 2000-01-03 00:00:00 -> 2025-05-30 00:00:00 | tickers: 33
NUMERICAL: 184 | GROWTH: 54 | TECH: 55 | PAT: 61 | CUST: ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'ln_volume'] | MACRO: ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS', 'DGS1', 'DGS5', 'DGS10']


C:\stock analytics 2026\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Q1 — Dummies de Month x Week-of-Month
Semana del mes con `(d.day - 1) // 7 + 1`; nueva categórica `month_wom` tipo `'October_w1'`; dummies con `pd.get_dummies`; correlación vs `is_positive_growth_30d_future` y ranking por `abs_corr`.

In [3]:
df["Month"] = pd.to_datetime(df["Month"]).dt.strftime("%B")
df["Weekday"] = df["Weekday"].astype(str)
df["week_of_month"] = (df["Date"].dt.day - 1) // 7 + 1
df["month_wom"] = df["Month"].astype(str) + "_w" + df["week_of_month"].astype(str)
CATEGORICAL = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]
dummies = pd.get_dummies(df[CATEGORICAL], dtype="int32")
DUMMIES = dummies.columns.to_list()
print("total dummies:", len(DUMMIES), "| month_wom dummies:", sum(c.startswith("month_wom_") for c in DUMMIES))
dfw = pd.concat([df, dummies], axis=1)

TO_PREDICT = [g for g in dfw.keys() if "future" in g]
corr = dfw[NUMERICAL + DUMMIES + TO_PREDICT].corr(numeric_only=True)[TARGET]
cdf = pd.DataFrame({"corr": corr}); cdf["abs_corr"] = cdf["corr"].abs()
mw = cdf.loc[[c for c in cdf.index if c.startswith("month_wom_")]].sort_values("abs_corr", ascending=False)
print(mw.head(10).to_string())
print("Q1 answer (rounded 3dp):", round(float(mw["abs_corr"].iloc[0]), 3), "| dummy:", mw.index[0])

total dummies: 115 | month_wom dummies: 60


                            corr  abs_corr
month_wom_September_w3 -0.030270  0.030270
month_wom_February_w3  -0.026895  0.026895
month_wom_October_w5    0.023846  0.023846
month_wom_March_w4      0.020862  0.020862
month_wom_June_w3      -0.017640  0.017640
month_wom_December_w3   0.016986  0.016986
month_wom_November_w4   0.016880  0.016880
month_wom_November_w3   0.016544  0.016544
month_wom_December_w4   0.016346  0.016346
month_wom_June_w1      -0.015874  0.015874
Q1 answer (rounded 3dp): 0.03 | dummy: month_wom_September_w3


## Split temporal + hand rules (Q2)
Split 70/15/15 por fecha como en Code Snippet 3; reglas manuales pred0-pred2 originales + pred3/pred4 del HW.

In [4]:
def temporal_split(d, min_date, max_date, train_prop=0.7, val_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    out = d.copy()
    out["split"] = np.where(out["Date"] <= train_end, "train", np.where(out["Date"] <= val_end, "validation", "test"))
    return out, train_end, val_end

new_df, train_end, val_end = temporal_split(dfw, dfw["Date"].min(), dfw["Date"].max())
print(new_df["split"].value_counts().to_dict(), "| train_end:", train_end.date(), "| val_end:", val_end.date())

new_df["pred0_manual_cci"] = (new_df["cci"] > 200).astype(int)
new_df["pred1_manual_prev_g1"] = (new_df["growth_1d"] > 1).astype(int)
new_df["pred2_manual_prev_g1_and_snp"] = ((new_df["growth_1d"] > 1) & (new_df["growth_snp500_1d"] > 1)).astype(int)
new_df["pred3_manual_dgs10_5"] = ((new_df["DGS10"] <= 4.5) & (new_df["DGS5"] <= 4)).astype(int)
new_df["pred4_manual_dgs10_fedfunds"] = ((new_df["DGS10"] > 4) & (new_df["FEDFUNDS"] <= 4.795)).astype(int)
PREDICTIONS = [k for k in new_df.keys() if k.startswith("pred")]
print(PREDICTIONS)
test = new_df[new_df["split"] == "test"]
for p in PREDICTIONS:
    print(p, "n_pos_test:", int(test[p].sum()), "precision:", round(float(precision_score(test[TARGET], test[p], zero_division=0)), 6))

{'train': 129481, 'test': 31359, 'validation': 30625} | train_end: 2017-10-15 | val_end: 2021-08-07
['pred0_manual_cci', 'pred1_manual_prev_g1', 'pred2_manual_prev_g1_and_snp', 'pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']
pred0_manual_cci n_pos_test: 793 precision: 0.540984
pred1_manual_prev_g1 n_pos_test: 16206 precision: 0.537147
pred2_manual_prev_g1_and_snp n_pos_test: 10852 precision: 0.539255
pred3_manual_dgs10_5 n_pos_test: 17836 precision: 0.53734
pred4_manual_dgs10_fedfunds n_pos_test: 5330 precision: 0.524765


## Q3 — Árbol depth=10 y predicciones únicamente correctas
Entrena en TRAIN+VALIDATION, predice todo el dataset en orden (`X_all`) y cuenta en TEST los casos donde `pred5_clf_10` acierta y `pred0..pred4` fallan.

In [5]:
features_list = NUMERICAL + DUMMIES
train_df = new_df[new_df["split"].isin(["train", "validation"])].copy()
test_df = new_df[new_df["split"] == "test"].copy()
X_train = train_df[features_list + ["Date", "Ticker", TARGET]].copy()
X_test = test_df[features_list + ["Date", "Ticker", TARGET]].copy()
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train.fillna(0, inplace=True); X_test.fillna(0, inplace=True)
y_train = X_train[TARGET].copy(); y_test = X_test[TARGET].copy()
Xtr = X_train.drop([TARGET, "Date", "Ticker"], axis=1)
Xte = X_test.drop([TARGET, "Date", "Ticker"], axis=1)
X_all = new_df[features_list + ["Date", "Ticker"]].copy()
X_all.replace([np.inf, -np.inf], np.nan, inplace=True); X_all.fillna(0, inplace=True)
X_all_nokey = X_all.drop(["Date", "Ticker"], axis=1)
print("X_train:", Xtr.shape, "| X_test:", Xte.shape, "| X_all:", X_all_nokey.shape)

clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(Xtr, y_train)
new_df["pred5_clf_10"] = clf10.predict(X_all_nokey)
print("pred5 TEST precision:", round(float(precision_score(new_df.loc[new_df.split == "test", TARGET], new_df.loc[new_df.split == "test", "pred5_clf_10"], zero_division=0)), 6))
hw = ["pred0_manual_cci", "pred1_manual_prev_g1", "pred2_manual_prev_g1_and_snp", "pred3_manual_dgs10_5", "pred4_manual_dgs10_fedfunds"]
new_df["only_pred5_is_correct"] = (((new_df["pred5_clf_10"] == new_df[TARGET])) & (~((new_df[hw] == new_df[TARGET].values[:, None]).any(axis=1)))).astype(int)
print("Q3 unique TEST:", int(new_df.loc[new_df.split == "test", "only_pred5_is_correct"].sum()))
print(export_text(clf10, feature_names=list(Xtr.columns), max_depth=2))

X_train: (160106, 299) | X_test: (31359, 299) | X_all: (191465, 299)


pred5 TEST precision: 0.544886
Q3 unique TEST: 2366
|--- gdppot_us_qoq <= 0.01
|   |--- fastd <= 0.25
|   |   |--- growth_brent_oil_90d <= 1.06
|   |   |   |--- class: 0
|   |   |--- growth_brent_oil_90d >  1.06
|   |   |   |--- truncated branch of depth 3
|   |--- fastd >  0.25
|   |   |--- growth_gold_365d <= 1.51
|   |   |   |--- truncated branch of depth 8
|   |   |--- growth_gold_365d >  1.51
|   |   |   |--- truncated branch of depth 8
|--- gdppot_us_qoq >  0.01
|   |--- DGS10 <= 4.38
|   |   |--- DGS5 <= 3.07
|   |   |   |--- truncated branch of depth 8
|   |   |--- DGS5 >  3.07
|   |   |   |--- truncated branch of depth 8
|   |--- DGS10 >  4.38
|   |   |--- growth_wti_oil_30d <= 0.99
|   |   |   |--- truncated branch of depth 8
|   |   |--- growth_wti_oil_30d >  0.99
|   |   |   |--- truncated branch of depth 8



## Q4 — Tuning max_depth 1..12
Mismo train (TRAIN+VALIDATION) y métrica (precisión TEST); reentrena el mejor en `pred6_clf_best`.

In [6]:
rows = []
for md in range(1, 13):
    clf = DecisionTreeClassifier(max_depth=md, random_state=42)
    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)
    rows.append((md, float(precision_score(y_test, yp, zero_division=0)), float(accuracy_score(y_test, yp)), int(yp.sum())))
res = pd.DataFrame(rows, columns=["max_depth", "precision_test", "accuracy_test", "n_pos_test"])
print(res.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
best = max(rows, key=lambda r: r[1])
print("Q4 best_max_depth:", best[0], "| precision:", round(best[1], 6))
clf_best = DecisionTreeClassifier(max_depth=best[0], random_state=42)
clf_best.fit(Xtr, y_train)
new_df["pred6_clf_best"] = clf_best.predict(X_all_nokey)
for p in ["pred0_manual_cci", "pred1_manual_prev_g1", "pred2_manual_prev_g1_and_snp", "pred3_manual_dgs10_5", "pred4_manual_dgs10_fedfunds", "pred5_clf_10", "pred6_clf_best"]:
    v = new_df.loc[new_df.split == "test", [p, TARGET]]
    print(p, round(float(precision_score(v[TARGET], v[p], zero_division=0)), 6))

 max_depth  precision_test  accuracy_test  n_pos_test
         1        0.537294       0.537294       31359
         2        0.537382       0.537453       31352
         3        0.542865       0.542460       29173
         4        0.540232       0.532702       27279
         5        0.537192       0.529258       28057
         6        0.536859       0.529386       28365
         7        0.536538       0.528046       28039
         8        0.545668       0.527313       22182
         9        0.535448       0.526643       28281
        10        0.544886       0.536879       25910
        11        0.545897       0.537326       25492
        12        0.543927       0.528716       23562
Q4 best_max_depth: 11 | precision: 0.545897


pred0_manual_cci 0.540984
pred1_manual_prev_g1 0.537147
pred2_manual_prev_g1_and_snp 0.539255
pred3_manual_dgs10_5 0.53734
pred4_manual_dgs10_fedfunds 0.524765
pred5_clf_10 0.544886
pred6_clf_best 0.545897


## Q5 — Qué datos faltan (exploratorio)
1. Régimen: VIX/vol realizada, HY OAS, pendiente 2s10s y sorpresas macro. 2. Divergencia regional RBI/ECB/Fed, FX USD/EUR/INR y pertenencia a índices (`ticker_type` es grueso). 3. Microestructura: turnover free-float, ATR normalizado, corporate actions y borrow-cost. 4. Forward fundamentals point-in-time: earnings yield vs DGS10, revisiones y calendario. 5. Alternativos: sentimiento (GDELT/RavenPack) y breadth sectorial. Fuentes: FRED/ECB/RBI, OHLCV actual, TA-Lib; Worldscope/Compustat o vendor para fundamentales.